> 在线讲义：[第 36 章 · 超节点的物理组织与内存池：数据走哪条线、借谁的内存](https://wikieden.github.io/ai-infra-from-constraints/lessons/36-topology-pool.html) · 本书《从约束推导 AI Infra》

# 第 36 章 · 超节点的物理组织与内存池：数据走哪条线、借谁的内存

配套讲义：仓库里 `ai-infra/notes/lessons/36-topology-pool.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



跨服务器 TP16、NVSwitch 端口分配、环面与交换网络、供电上限、UB 连接状态、内存池的在途事务取自 AII 第 6 章 §6.5–§6.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KiB, MiB = 2**10, 2**20

**函数 `tp16_cross_server`**：小工具。

In [ ]:
def tp16_cross_server(local_ms: float = 2.48, ar_us: float = 32.74) -> float:
    return local_ms + 128 * ar_us / 1e3

**函数 `uplink_gbps`**：小工具。

In [ ]:
def uplink_gbps(up_ports: int, port_gbps: float = 25) -> float:
    return up_ports * port_gbps

**函数 `torus_alltoall_us`**：均匀 All-to-All：(环面每条有向链路负载对应的时间, 交换网络每端口时间)。平均跳数 = 维数 × k/4。

In [ ]:
def torus_alltoall_us(M: float, k: int = 4, ports: int = 6, port_bw: float = 50e9) -> tuple[float, float]:
    """均匀 All-to-All：(环面每条有向链路负载对应的时间, 交换网络每端口时间)。平均跳数 = 维数 × k/4。"""
    n = k**3
    hops = 3 * k / 4
    links = n * ports
    torus = hops * n * M / links / port_bw
    switch = M / ports / port_bw
    return torus * 1e6, switch * 1e6

**函数 `max_cards`**：式 (6-11)。

In [ ]:
def max_cards(budget_kw: float, server_kw: float = 10.2, g: int = 8, fixed_kw: float = 0) -> int:
    """式 (6-11)。"""
    return g * math.floor((budget_kw - fixed_kw) / server_kw)

**函数 `nic_state`**：(逐对连接, UB 端点加通道) 的每网卡状态字节。

In [ ]:
def nic_state(H: int, A: int = 8) -> tuple[int, int]:
    """(逐对连接, UB 端点加通道) 的每网卡状态字节。"""
    return 512 * A * A * (H - 1) + 32 * A, 52 * A + 56 * (H - 1)

**函数 `max_hosts`**：小工具。

In [ ]:
def max_hosts(cache: int = 256 * KiB, A: int = 8) -> tuple[int, int]:
    pair = (cache - 32 * A) // (512 * A * A) + 1
    ub = (cache - 52 * A) // 56 + 1
    return pair, ub

**函数 `setup_s`**：小工具。

In [ ]:
def setup_s(N: int, M: int, syscall: float = 5e-6, rtt: float = 500e-6, cores: int = 32) -> tuple[float, float]:
    pair = N * M * (4 * syscall + rtt)
    ub = N * syscall + M * (syscall + rtt)
    return pair / cores, ub / cores

**函数 `window_bw`**：式 (6-12)。

In [ ]:
def window_bw(u: int, q: int = 256, L: float = 7.52e-6, path: float = 50e9) -> float:
    """式 (6-12)。"""
    return min(path, u * q / L)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：再加一台服务器")
print(f"  TP16 跨服务器 {tp16_cross_server():.2f} ms（TP8 机内 5.15 ms）；同一 NVLink 域 {2.48 + 128 * 30 * 0.822e-3:.2f} ms")
print(f"  NVSwitch 64 端口：32 / 32 上联 {uplink_gbps(32):.0f} GB/s，48 / 16 上联 {uplink_gbps(16):.0f} GB/s")

In [ ]:
print("\n第二步：64 卡环面 vs 交换网络（每卡 6 × 50 GB/s，每卡发 32 MiB）")
t, s = torus_alltoall_us(32 * MiB)
print(f"  All-to-All：环面 {t:.0f} μs，交换 {s:.0f} μs；512 卡环面 {torus_alltoall_us(32 * MiB, k=8)[0]:.0f} μs")
print(f"  环面最远多 4 颗中间芯片 × 200 ns = 0.8 μs，128 次 {128 * 0.8 / 1e3:.2f} ms")
for k in (4, 8):
    print(f"  {k}³ 环面：{k**3} 颗芯片，二分链路 {2 * k * k} 条")
print(f"  Swing vs 递归 前三轮瓶颈：{12 * MiB / 50e9 * 1e6:.0f} vs {8 * MiB / 50e9 * 1e6:.0f} μs")

In [ ]:
print("\n第三步：供电")
for n in (64, 72, 96):
    print(f"  {n} 卡 {n / 8 * 10.2:.1f} kW")
print(f"  120 kW 最多 {max_cards(120)} 张卡")

In [ ]:
print("\n第四步：UB 的连接状态")
pr, ub = nic_state(192)
print(f"  192 台主机：逐对 {pr / MiB:.1f} MiB，端点加通道 {ub / KiB:.1f} KiB；256 KiB 缓存最多 {max_hosts()} 台")
a, b = setup_s(1024, 1024)
print(f"  N = M = 1024 建连（32 核）：逐对 {a:.0f} s，UB {b * 1e3:.0f} ms")

In [ ]:
print("\n第五步：内存池")
print(f"  128 个在途 × 256 B / 7.52 μs = {window_bw(128) / 1e9:.2f} GB/s；用满 50 GB/s 要 {math.ceil(50e9 * 7.52e-6 / 256)} 个；4096 个 → {4096 * 256 / 7.52e-6 / 1e9:.0f} GB/s")
print(f"  20 GB：50 GB/s {20e9 / 50e9:.2f} s，4.36 GB/s {20e9 / window_bw(128):.2f} s；读 10 次：远端 {10 * 20e9 / 50e9:.1f} s，取回 + 本地 {20e9 / 50e9 + 10 * 20e9 / 3350e9:.2f} s")

## 3. 代码里才看得到的

1. 环面平均跳数按每维 $k/4$ 算（$k$ 为偶数时环上平均距离），三维共 $3k/4$。
2. 交换网络按无阻塞算：交换层不限制各端口同时满速。
3. 建连时间里，UB 的 $M$ 个远端各要一次带外往返，所以对往返时延也敏感，但只乘 $M$ 而不是 $NM$。

## 4. 自己改一改

- 给 `torus_alltoall_us` 加二维环面版本，比较 8×8 和 4×4×4。
- 把 `nic_state` 的缓存改成 1 MiB、端点改成 64，算逐对连接能容纳几台。
- 用 `window_bw` 画出往返 2、7.52、20 μs 三条曲线。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 多叫外地同事，材料分得少，表决要等电话；
> 大门通道分给住户多，通往大路就少；
> 棋盘街去远处穿好几户，高速任意两家上下一次；规则的活棋盘不吃亏；
> 电闸定空调数，门禁表随住户长；
> 借邻居书柜，往返一趟能带几本决定查得多快。